In [1]:
#pip install pm4py
import os
os.getcwd()

'C:\\Users\\obami\\Documents\\Python_Pra\\Thesis_PPM_2024-25'

In [2]:
#import and preprocess data
import numpy as np
import pandas as pd
import pm4py
import joblib

#Enode Prefix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from keras.preprocessing.sequence import pad_sequences

1. Load data and keep necessary columns

In [3]:
df = pd.read_csv("dmd_df.csv")
#df = pd.read_csv("ptc_df.csv")
#df = pd.read_csv('helpdesk_df.csv')
df.head()

,timestamp,activity,case_id
0,2017-01-09 09:49:50+00:00,declaration submitted by employee,declaration 86791
1,2017-01-09 10:26:14+00:00,declaration submitted by employee,declaration 86795
2,2017-01-09 11:13:33+00:00,declaration submitted by employee,declaration 86800
3,2017-01-09 11:24:20+00:00,declaration submitted by employee,declaration 86731
4,2017-01-09 11:27:48+00:00,declaration final_approved by supervisor,declaration 86791


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56437 entries, 0 to 56436
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   timestamp  56437 non-null  object
 1   activity   56437 non-null  object
 2   case_id    56437 non-null  object
dtypes: object(3)
memory usage: 1.3+ MB


In [5]:
df["timestamp"] = pd.to_datetime(df["timestamp"]) # conversion from object to date type

2. Split data into train-validation-test

In [6]:
#split data test
from split_train_test import split_train_test_temporal
train, test, fp_dict = split_train_test_temporal(df,0.2,"case_id","timestamp","preferred")

#split data validation
train, val, fp_dict = split_train_test_temporal(train,0.1,"case_id","timestamp","preferred")

3. Add BOS and EOS¶


add bos and eos at the end of every case for prefix
add eos at the end of every case for target

In [7]:
from bos_eos import add_bos_eos_target
############## train data transformation ########################
train_prefix = add_bos_eos_target(train,"prefix")
train_target = add_bos_eos_target(train)

############## validation data transformation ###################
val_prefix = add_bos_eos_target(val,"prefix")
val_target = add_bos_eos_target(val)

############## test data transformation ########################
test_prefix = add_bos_eos_target(test,"prefix")
test_target = add_bos_eos_target(test)

4. Create Prefix Trace

In [8]:
from prefix_trace import prefix_trace
############## train data transformation ########################
train_prefix_trace = prefix_trace(train_prefix)

############## validation data transformation ###################
val_prefix_trace = prefix_trace(val_prefix)

############## test data transformation ########################
test_prefix_trace = prefix_trace(test_prefix)

5. Prefix Simple Index Encoding

In [9]:
from encoding import find_max_len, si_encoding

#all possible cases. I assume in a business all activities are already known and defined
cases = df["activity"].unique()
cases = np.append(cases,"BOS")
cases = np.append(cases,"zos")

#find the maximum length of the longest case for padding
max_len = find_max_len(train_prefix_trace["prefix"],val_prefix_trace["prefix"],test_prefix_trace["prefix"])

from encoding import si_encoding
############## train data transformation ########################
train_prefix_trace_encoded, label_encoder = si_encoding(train_prefix_trace,cases,max_len)
train_target_encoded, a = si_encoding(train_target,cases,option = "target")

In [10]:
############## validation data transformation ###################
val_prefix_trace_encoded, a = si_encoding(val_prefix_trace,cases,max_len)
val_target_encoded, a = si_encoding(val_target,cases,option="target")

In [11]:
############## test data transformation ########################
test_prefix_trace_encoded, a = si_encoding(test_prefix_trace,cases,max_len)
test_target_encoded, a = si_encoding(test_target,cases,option="target")

6. Playout Probabilities

In [12]:
from dfg_probabilities import dfg_df

############ train #################
#get probability
train_prefix_copy = train_prefix.copy()
probability = dfg_df(train_prefix,cases)

#encode labels for probability index
probability.index = label_encoder.transform(probability.index)
probability.columns = label_encoder.transform(probability.columns)

#reset index
probability.reset_index(inplace=True)
probability.rename(columns = {"index":"activity"},inplace=True)

#encode drop extra columns and encode activity
train_prefix["activity"] = label_encoder.transform(train_prefix["activity"])

#merge to get new dataframe
train_dfg_probability = pd.merge(train_prefix,probability,how="left",on="activity")
train_dfg_probability = train_dfg_probability.drop(columns = ["timestamp","case_id"])

In [13]:
############ validation ################################## 
#encode drop extra columns and encode activity
val_prefix["activity"] = label_encoder.transform(val_prefix["activity"])

#merge to get new dataframe. probability is same as train
val_dfg_probability = pd.merge(val_prefix,probability,how="left",on="activity")
val_dfg_probability = val_dfg_probability.drop(columns = ["timestamp","case_id"])

In [14]:
############ test ################################## 
#probability is combination of train and validation
train_val_prefix = pd.concat([train_prefix_copy,val_prefix])
probability = dfg_df(train_val_prefix,cases)

#encode drop extra columns and encode activity
test_prefix["activity"] = label_encoder.transform(test_prefix["activity"])

#encode labels for probability index
probability.index = label_encoder.transform(probability.index)
probability.columns = label_encoder.transform(probability.columns)

#reset index
probability.reset_index(inplace=True)
probability.rename(columns = {"index":"activity"},inplace=True)

#merge to get new dataframe
test_dfg_probability = pd.merge(test_prefix,probability,how="left",on="activity")
test_dfg_probability = test_dfg_probability.drop(columns = ["timestamp","case_id"])

Save Data

In [15]:
########################################### Help Desk ###########################################################################

##prefix data
np.save("helpdesk_train_prefix.npy",train_prefix_trace_encoded)
np.save("helpdesk_val_prefix.npy",val_prefix_trace_encoded)
np.save("helpdesk_test_prefix.npy",test_prefix_trace_encoded)

##probability data
train_dfg_probability.to_csv("helpdesk_train_dfg_probability.csv",index=False)
val_dfg_probability.to_csv("helpdesk_val_dfg_probability.csv",index=False)
test_dfg_probability.to_csv("helpdesk_test_dfg_probability.csv",index=False)

#target
np.save("helpdesk_train_target.npy",train_target_encoded)
np.save("helpdesk_val_target.npy",val_target_encoded)
np.save("helpdesk_test_target.npy",test_target_encoded)

#original data
train_target.to_csv("helpdesk_train_target_org.csv",index=False)
test_target.to_csv("helpdesk_test_target_org.csv",index=False)

In [14]:
######################################### BPI Data ###############################################################################

##prefix data
np.save("ptc_train_prefix.npy",train_prefix_trace_encoded)
np.save("ptc_val_prefix.npy",val_prefix_trace_encoded)
np.save("ptc_test_prefix.npy",test_prefix_trace_encoded)

##probability data
train_dfg_probability.to_csv("ptc_train_dfg_probability.csv",index=False)
val_dfg_probability.to_csv("ptc_val_dfg_probability.csv",index=False)
test_dfg_probability.to_csv("ptc_test_dfg_probability.csv",index=False)

#target
np.save("ptc_train_target.npy",train_target_encoded)
np.save("ptc_val_target.npy",val_target_encoded)
np.save("ptc_test_target.npy",test_target_encoded)

#original data
train_target.to_csv("ptc_train_target_org.csv",index=False)
test_target.to_csv("ptc_test_target_org.csv",index=False)

In [14]:
######################################### RMP Data ###############################################################################

##prefix data
np.save("dmd_train_prefix.npy",train_prefix_trace_encoded)
np.save("dmd_val_prefix.npy",val_prefix_trace_encoded)
np.save("dmd_test_prefix.npy",test_prefix_trace_encoded)

##probability data
train_dfg_probability.to_csv("dmd_train_dfg_probability.csv",index=False)
val_dfg_probability.to_csv("dmd_val_dfg_probability.csv",index=False)
test_dfg_probability.to_csv("dmd_test_dfg_probability.csv",index=False)

#target
np.save("dmd_train_target.npy",train_target_encoded)
np.save("dmd_val_target.npy",val_target_encoded)
np.save("dmd_test_target.npy",test_target_encoded)

#original data
train_target.to_csv("dmd_train_target_org.csv",index=False)
test_target.to_csv("dmd_test_target_org.csv",index=False)

In [29]:
# Helpdesk 
joblib.dump(label_encoder, 'helpdesk_label_encoder.joblib')

['helpdesk_label_encoder.joblib']

In [15]:
# dmd 
joblib.dump(label_encoder, 'dmd_label_encoder.joblib')

['dmd_label_encoder.joblib']

In [15]:
# ptc 
#joblib.dump(label_encoder, 'ptc_label_encoder.joblib')

['ptc_label_encoder.joblib']

In [53]:
dfg_df


<function dfg_probabilities.dfg_df(df_prefix, cases)>

In [45]:
print(dfg_df)

<function dfg_df at 0x000002331EE1A160>


In [34]:
#df = pd.read_csv("dmd_df.csv")
#df = pd.read_csv("ptc_df.csv")
df1 = pd.read_csv('helpdesk_df.csv')
df1.head()

,timestamp,activity,case_id
0,2010-01-13 08:40:25+00:00,assign seriousness,Case3608
1,2010-01-13 12:26:04+00:00,assign seriousness,Case2748
2,2010-01-13 12:30:37+00:00,assign seriousness,Case4284
3,2010-01-13 13:09:31+00:00,assign seriousness,Case1534
4,2010-01-13 17:25:25+00:00,assign seriousness,Case406


In [35]:
len(df1["activity"].unique()) # Checking the number of activityin the dataset

14

In [36]:
(df1["activity"].value_counts ()/ df1.shape[0]) * 100 # Helpdesk -Checking the distribution

activity
take in charge ticket    23.834881
assign seriousness       23.226992
resolve ticket           23.066773
closed                   21.492861
wait                      6.832854
insert ticket             0.556053
require upgrade           0.556053
create sw anomaly         0.311013
resolve sw anomaly        0.061260
schedule intervention     0.023562
verified                  0.014137
resolved                  0.009425
invalid                   0.009425
duplicate                 0.004712
Name: count, dtype: float64

In [39]:

#split data test
from split_train_test import split_train_test_temporal
train1, test1, fp_dict = split_train_test_temporal(df1,0.2,"case_id","timestamp","preferred")

#split data validation
train1, val1, fp_dict = split_train_test_temporal(train1,0.1,"case_id","timestamp","preferred")

In [40]:
(train1["activity"].value_counts ()/ train1.shape[0]) * 100 # Helpdesk - Checking the distribution

activity
take in charge ticket    25.139043
assign seriousness       23.767149
resolve ticket           23.018168
closed                   21.334816
wait                      5.591398
insert ticket             0.778643
create sw anomaly         0.281795
resolve sw anomaly        0.037078
schedule intervention     0.029663
resolved                  0.007416
invalid                   0.007416
verified                  0.007416
Name: count, dtype: float64

In [41]:
len(train1["activity"].unique())

12

In [4]:
#df = pd.read_csv("dmd_df.csv")
df2 = pd.read_csv("ptc_df.csv")
#df2 = pd.read_csv('helpdesk_df.csv')
df2.head()

,timestamp,activity,case_id
0,2017-01-09 14:48:43,permit submitted by employee,request for payment 73550
1,2017-01-09 14:48:43,permit submitted by employee,request for payment 73552
2,2017-01-09 14:48:55,permit final_approved by supervisor,request for payment 73550
3,2017-01-09 14:48:55,permit final_approved by supervisor,request for payment 73552
4,2017-01-10 11:19:16,permit submitted by employee,request for payment 76316


In [5]:
#split data test
from split_train_test import split_train_test_temporal
train2, test2, fp_dict = split_train_test_temporal(df2,0.2,"case_id","timestamp","preferred")

#split data validation
train2, val2, fp_dict = split_train_test_temporal(train2,0.1,"case_id","timestamp","preferred")

In [7]:
(train2["activity"].value_counts ()/ train2.shape[0]) * 100 # PTC Checking the distribution

activity
request for payment submitted by employee           12.448000
request payment                                     11.338667
payment handled                                     11.328000
permit submitted by employee                        11.210667
request for payment final_approved by supervisor    11.189333
permit final_approved by supervisor                  8.288000
request for payment approved by administration       8.288000
permit approved by administration                    7.605333
request for payment approved by budget owner         3.402667
permit approved by budget owner                      2.709333
permit final_approved by director                    2.464000
permit approved by supervisor                        2.464000
request for payment approved by pre_approver         1.706667
permit approved by pre_approver                      1.685333
request for payment rejected by employee             1.024000
request for payment rejected by administration       0.725333